# Reproduce Table 3 — Ablation Study

**Paper:** Section 7.4  
**Prerequisite:** `scripts/reproduce/run_ablations.sh` must be complete.

Loads ablation experiment logs, computes mean ± std over 10 seeds,
and reproduces Table 3 with delta-MAP relative to the full GeoNet model.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_ROOT = Path('outputs')
SEEDS       = list(range(10))
VARIANTS    = [
    ('geonet',          'GeoNet (full)'),
    ('geonet_no_hel',   'GeoNet-noHEL'),
    ('geonet_no_gaa',   'GeoNet-noGAA'),
    ('geonet_no_rom',   'GeoNet-noROM'),
    ('geonet_fixed_c',  'GeoNet-fixed-c'),
    ('geonet_eucl',     'GeoNet-Eucl'),
]
DATASET = 'wordnet_mammals'

def load_metric(model, dataset, metric, seeds=SEEDS):
    vals = []
    for seed in seeds:
        p = OUTPUT_ROOT / f'{model}_{dataset}_seed{seed}' / 'results.json'
        if p.exists():
            with open(p) as f:
                r = json.load(f)
            if metric in r['test_metrics']:
                vals.append(r['test_metrics'][metric])
    return np.array(vals)

# Reference: full GeoNet
geonet_map = load_metric('geonet', DATASET, 'MAP')
print(f'GeoNet MAP: {geonet_map.mean():.4f} ± {geonet_map.std():.4f}  (n={len(geonet_map)})')

In [ ]:
rows = []
for model_id, model_name in VARIANTS:
    map_vals  = load_metric(model_id, DATASET, 'MAP')
    dist_vals = load_metric(model_id, DATASET, 'distortion')
    conv_vals = load_metric(model_id, DATASET, 'best_epoch')  # convergence epoch

    mean_map  = map_vals.mean()  if len(map_vals)  else np.nan
    std_map   = map_vals.std()   if len(map_vals)  else np.nan
    mean_dist = dist_vals.mean() if len(dist_vals) else np.nan
    mean_conv = conv_vals.mean() if len(conv_vals) else np.nan
    delta_map = mean_map - geonet_map.mean()

    rows.append({
        'Variant':        model_name,
        'MAP':            f'{mean_map:.3f} ± {std_map:.3f}',
        'Distortion':     f'{mean_dist:.3f}',
        'Delta MAP':      f'{delta_map:+.3f}',
        'Converge (ep.)': f'{mean_conv:.0f}' if not np.isnan(mean_conv) else 'N/A',
        'n':              len(map_vals),
    })

df = pd.DataFrame(rows)
print('\n=== Table 3: Ablation Study (WordNet-Mammals) ===')
print(df.to_string(index=False))

import os; os.makedirs('outputs/tables', exist_ok=True)
df.to_csv('outputs/tables/table3_ablations.csv', index=False)
print('\nSaved to outputs/tables/table3_ablations.csv')

In [ ]:
# Bar chart of Delta MAP per variant
import matplotlib.pyplot as plt

names  = [r['Variant']  for r in rows]
deltas = [float(r['Delta MAP']) for r in rows]
colors = ['#2E618A' if d == 0 else ('#E05C5C' if d < 0 else '#5CBE6A') for d in deltas]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, deltas, color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Delta MAP vs. Full GeoNet', fontsize=12)
ax.set_title('Table 3: Ablation Study — WordNet-Mammals MAP', fontsize=13)
for bar, val in zip(bars, deltas):
    ax.text(val - 0.001 if val < 0 else val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha='right' if val < 0 else 'left', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/tables/table3_ablations_chart.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')